# Reporting usage examples

This page contains examples for using the reporting module.

[See full module documentation here](project:/apidocs/pogg/pogg.evaluation.reporting.md)

The reporting module contains the following classes, each containing static functions that help in building readable reports about the results of running the POGG data-to-text algorithm on a dataset:

- `POGGGraphReporting` -- functions that help build tables detailing evaluation results about individual graphs
- `POGGDatasetReporting` -- functions that help build tables detailing evaluation results about an entire dataset

For the most part, interacting directly with reporting classes is not necessary, and the POGG data-to-text algorithm will print reports about each graph as well as a summary about the whole dataset as part of its process. But this page will explain what happens under the hood for each of these classes and how their functions contribute to the final reports.

## Sample Dataset

For these examples, we will use a small sample dataset with the following two graphs:

<img src="../../../data/images/cake_graph.png" alt="Directed graph with a root node labeled 'cake' with an edge labeled 'flavor' pointing to a child node labeled 'vanilla' and another edge labeled 'filling' pointing to a child node labeled 'raspberry'. The node labeled 'raspberry' has an edge labeled 'state' pointing to a child node labeled 'fresh'." width="40%" height="auto">

<img src="../../../data/images/cupcake_graph.png" alt="Directed graph with a root node labeled 'cupcake' with an edge labeled 'flavor' pointing to a child node labeled 'vanilla'." width="40%" height="auto">

We will also assume we have the following lexicon:

 :::{code} Lexicon
:collapsible:

```json
{
    "node_keys": {
        "cake": {
            "comp_fxn": "noun",
            "predicate": "_cake_n_1",
            "intrinsic_variable_properties": {}
        },
        "raspberry": {
            "comp_fxn": "noun",
            "predicate": "_raspberry_n_1",
            "intrinsic_variable_properties": {}
        },
        "vanilla": {
            "comp_fxn": "noun",
            "predicate": "_vanilla_n_1",
            "intrinsic_variable_properties": {}
        }
    },
    "edge_keys": {
        "flavor": {
            "comp_fxn": "compound_noun",
            "head_noun_sement": "parent",
            "non_head_noun_sement": "child"
        },
        "state": {
            "comp_fxn": "prenominal_adjective",
            "adjective_sement": "child",
            "nominal_sement": "parent"
        }
    }
}
```
:::

Before we can get to reporting, let's run the datat-to-text algorithm on the dataset.

In [3]:
from pogg.pogg_routine import POGG
pogg_obj = POGG("../config.yml", "../toy_dataset.yml")

# run data-to-text algorithm
pogg_obj.run_POGG_data_to_text_algorithm()

Converting ../../../data/just_a_cake_dataset/BitsyBakery/graph_jsons/vanilla_cake.json...
Converting ../../../data/just_a_cake_dataset/BitsyBakery/graph_jsons/vanilla_cupcake.json...


NOTE: 131 passive, 305 active edges in final generation chart; built 188 passives total. [12 results]
NOTE: generated 1 / 1 sentences, avg 3656k, time 0.35319s
NOTE: transfer did 72 successful unifies and 121 failed ones


Now for each of these graphs let's look at the generated SEMENT (if it exists) and the text results (if they exist)

In [4]:
# get the evaluation objects to look at the results
first_graph_eval = pogg_obj.evaluation.get_graph_evaluation("vanilla_cake")
second_graph_eval = pogg_obj.evaluation.get_graph_evaluation("vanilla_cupcake")

# look at the SEMENT and text results for the first graph
print(f"vanilla_cake generated SEMENT string:\n{first_graph_eval.generated_SEMENT_string}")

# print English text results
print("vanilla_cake text results:\n")
for result in first_graph_eval.generated_results:
    print(result)

# look at the SEMENT and text results for the second graph
print(f"\nvanilla_cupcake generated SEMENT string:\n{second_graph_eval.generated_SEMENT_string}")

# print English text results
print("vanilla_cupcake text results:\n")
for result in second_graph_eval.generated_results:
    print(result)

vanilla_cake generated SEMENT string:
[ TOP: h2
  INDEX: x1
  RELS: < [ _cake_n_1 LBL: h2 ARG0: x1 ] > ]
vanilla_cake text results:

Cake
The cakes.
The cakes
A cake
Cakes
Cake.
A cake.
The cake.
Cakes.
The cake
Cake.
Cake

vanilla_cupcake generated SEMENT string:
None
vanilla_cupcake text results:



Looking at the SEMENTs and text results show us that only the first graph successfully generated a SEMENT and text results. This is the same toy dataset that is used for the examples in the [evaluation usage guide](project:/usage_nbs/pogg/evaluation/evaluation_usage.ipynb), so for a deeper look into how the evaluation takes place see that page.

Now that we have evaluation objects with information in them from running the data-to-text algorithm, let's look at the ways each of the reoprting classes contributes to our final readable reports.

## `POGGGraphReporting` functions

Functions in the `POGGGraphReporting` class has functions for building tables about evaluation metrics for individual graphs:

| Function | Description |
|----------|-------------|
| `build_ASCII_nodes_table` | build a table detailing node-level evaluation metrics about all nodes in a graph |
| `build_ASCII_edges_table` | build a table detailing edge-level evaluation metrics about all edges in a graph |
| `build_ASCII_graph_metrics_table` | build a table detailing graph-level evaluation |
| `build_ASCII_graph_sement_table` | build a table showing the SEMENTs associated with a particular graph |
| `built_ASCII_graph_generated_text_table` | build a table showing the generated English text results that the ERG generated for the SEMENT associated with the graph |
| `build_ASCII_graph_report_detail` | build a full report for an individual graph that includes tables generated by all the functions listed above |

Let's look at each of the tables in sequence for the first graph (`vanilla_cake`) in our toy dataset:

### `build_ASCII_nodes_table`

This table shows us whether each node in the graph was covered (i.e. generated a SEMENT), included (i.e. contributed semantically to the final SEMENT for the graph), a comment describing the failure to be covered/included where applicable, and the generated SEMENT for the node where applicable.

In [5]:
from pogg.evaluation.reporting import POGGGraphReporting

print(POGGGraphReporting.build_ASCII_nodes_table(first_graph_eval))

+----------------------------------------------------------------------------------------------------------------------+
|                                                     NODE METRICS                                                     |
+-----------+--------------+---------------+---------------------------------------+-----------------------------------+
| Node      | Node Covered | Node Included | Coverage Comment                      | Inclusion Comment                 |
+-----------+--------------+---------------+---------------------------------------+-----------------------------------+
| cake      | True         | True          | Covered                               | Included                          |
| fresh     | False        | False         | 'fresh' not in lexicon's node entries | No SEMENT generated for this node |
| raspberry | True         | False         | Covered                               | Successor of failed element       |
| vanilla   | True         | Fal

### `build_ASCII_edges_table`

This table shows us whether each edge in the graph was covered (i.e. generated a SEMENT), included (i.e. contributed semantically to the final SEMENT for the graph), a comment describing the failure to be covered/included where applicable, and the generated SEMENT for the edge where applicable.

In [6]:
print(POGGGraphReporting.build_ASCII_edges_table(first_graph_eval))

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|                                                                                                                                                                 EDGE METRICS                                                                                                                                                                |
+---------+-----------+-----------+--------------+---------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------

### `build_ASCII_graph_metrics_table`

This table shows us the quantitative metrics of total text results, node coverage, node inclusion, edge coverage, and edge inclusion for the graph as a whole.

In [7]:
print(POGGGraphReporting.build_ASCII_graph_metrics_table(first_graph_eval))

+-----------------------------------------+
|            EVALUATION METRICS           |
+---------------------------------+-------+
| Metric                          | Value |
+---------------------------------+-------+
| Generated Text Results          | 12    |
| Generated Gold Results          | 0     |
| Generated Gold Results Coverage | 0.0   |
+---------------------------------+-------+
| Node Count                      | 4     |
| Nodes Covered                   | 3     |
| Nodes Included                  | 1     |
+---------------------------------+-------+
| Node Coverage                   | 0.75  |
| Node Inclusion                  | 0.25  |
+---------------------------------+-------+
| Edge Count                      | 3     |
| Edges Covered                   | 0     |
| Edges Included                  | 0     |
+---------------------------------+-------+
| Edge Coverage                   | 0.0   |
| Edge Inclusion                  | 0.0   |
+-------------------------------

### `build_ASCII_graph_SEMENT_table`

This table shows the actual SEMENT and its variations that were generated for the graph, if any.

In [8]:
print(POGGGraphReporting.build_ASCII_graph_SEMENT_table(first_graph_eval))

+--------------------------------------------------------------------------------+
|                               GENERATED SEMENTS                                |
+---------------+----------------------------------------------------------------+
| Version       | SEMENT                                                         |
+---------------+----------------------------------------------------------------+
| Original      | [ TOP: h2                                                      |
|               |   INDEX: x1                                                    |
|               |   RELS: < [ _cake_n_1 LBL: h2 ARG0: x1 ] > ]                   |
+---------------+----------------------------------------------------------------+
| EQs Collapsed | [ TOP: h2                                                      |
|               |   INDEX: x1                                                    |
|               |   RELS: < [ _cake_n_1 LBL: h2 ARG0: x1 ] > ]                   |
+---

### `build_ASCII_graph_generated_text_table`

This table shows the list of generated English text results that the ERG generated from the SEMENT, if any.

In [9]:
print(POGGGraphReporting.build_ASCII_graph_generated_text_table(first_graph_eval))

+------------------------+
| GENERATED ENGLISH TEXT |
+----+-------------------+
| #  | Generated Text    |
+----+-------------------+
| 1  | A cake            |
| 2  | A cake.           |
| 3  | Cake              |
| 4  | Cake              |
| 5  | Cake.             |
| 6  | Cake.             |
| 7  | Cakes             |
| 8  | Cakes.            |
| 9  | The cake          |
| 10 | The cake.         |
| 11 | The cakes         |
| 12 | The cakes.        |
+----+-------------------+


### `build_ASCII_graph_report_detail`

The last function in the `POGGGraphReporting` class prints all the tables showed above into a report named after the graph to the `evaluation` directory specified by the user in their configuration file for their dataset.

Now let's move onto the tables we can generate for the whole dataset.

## `POGGDatasetReporting`

Functions in the `POGGDatasetReporting` class has functions for building tables about evaluation metrics for the whole dataset:

| Function | Description |
|----------|-------------|
| `build_ASCII_dataset_metrics_table` | build an ASCII table detailing evaluation metrics for a dataset |
| `build_ASCII_graphs_report_summary` | build an ASCII table detailing evaluation metrics for each graph in the dataset |
| `build_ASCII_dataset_report` | build a report for the dataset that includes both of the above tables |

Let's look at each of the tables in sequence for the toy dataset.

### `build_ASCII_dataset_metrics_table`

This table shows the dataset-level quantitative metrics for the dataset.

In [10]:
from pogg.evaluation.reporting import POGGDatasetReporting

print(POGGDatasetReporting.build_ASCII_dataset_metrics_table(pogg_obj.evaluation))

+------------------------------------------------------------------+
|                        EVALUATION METRICS                        |
+--------------------------------------------+---------------------+
| Metric                                     | Value               |
+--------------------------------------------+---------------------+
| Graph Count                                | 2                   |
| Graphs that produced SEMENTs               | 1                   |
| Graphs w/ SEMENTs coverage                 | 0.5                 |
| Graphs that generated text                 | 1                   |
| Graphs w/ text coverage                    | 0.5                 |
+--------------------------------------------+---------------------+
| Total Node Count                           | 6                   |
| Total Nodes Covered                        | 4                   |
| Total Nodes Included                       | 1                   |
| Total Node Coverage             

### `build_ASCII_graphs_report_summary`

This table shows the graph-level evaluation metrics for each graph in the dataset.

In [11]:
print(POGGDatasetReporting.build_ASCII_graphs_report_summary(pogg_obj.evaluation))

+-------------------------------------------------------------------------------------------------------------------------------------------+
|                                                              GRAPH SUMMARIES                                                              |
+-----------------+-------------------+-------------------+-------+---------------+----------------+-------+---------------+----------------+
| Graph Name      | Generated SEMENT? | # of text results | Nodes | Node Coverage | Node Inclusion | Edges | Edge Coverage | Edge Inclusion |
+-----------------+-------------------+-------------------+-------+---------------+----------------+-------+---------------+----------------+
| vanilla_cake    | True              | 12                | 4     | 0.75          | 0.25           | 3     | 0.0           | 0.0            |
| vanilla_cupcake | False             | 0                 | 2     | 0.5           | 0.0            | 1     | 0.0           | 0.0            |
+-----

### `build_ASCII_dataset_report`

This function prints the two dataset-level tables showed above into a report in the `evaluation` directory specified by the user in their configuration file for their dataset.

## Conclusion

Most users will not be directly printing report tables as we did in this guide, but rather the data-to-text algorithm will print a report for each graph in the dataset as well as an overall report of the dataset to the `evaluation` directory specified in the dataset configuration file. 